<a href="https://colab.research.google.com/github/DaniChinwendu/AGENTS-FOR-BIOLOGY/blob/main/pubhomics_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

filename = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {filename}")

Saving GEO_9.txt to GEO_9.txt

✅ Uploaded: GEO_9.txt


In [ ]:
import math

def split_file(filename, parts=5):
    try:
        # 1. Read the content of the file
        with open(filename, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        total_lines = len(lines)
        lines_per_part = math.ceil(total_lines / parts)

        print(f"Total lines: {total_lines}")
        print(f"Lines per part (approx): {lines_per_part}")

        # 2. Split and write to new files
        for i in range(parts):
            start_index = i * lines_per_part
            end_index = start_index + lines_per_part

            # Slice the list of lines for this part
            chunk = lines[start_index:end_index]

            # Create a name for the new file (e.g., GEO 4_part1.txt)
            part_filename = f"{filename.replace('.txt', '')}_part{i+1}.txt"

            with open(part_filename, 'w', encoding='utf-8') as f_out:
                f_out.writelines(chunk)

            print(f"Created {part_filename} with {len(chunk)} lines.")

    except FileNotFoundError:
        print(f"Error: The file '{filename}' was not found.")

# Usage: Replace 'GEO 4.txt' with the actual path to your file
split_file('GEO 4.txt')

Error: The file 'GEO 4.txt' was not found.


In [ ]:
# ============================================================================
# Cell 1: Install Dependencies
# ============================================================================
print("📦 Installing dependencies...")
import subprocess

packages = [
    'langchain>=0.3.0',
    'langchain-core>=0.3.0',
    'langchain-google-genai>=2.0.0',
    'langgraph>=0.2.0',
    'pydantic>=2.0',
    'pandas',
    'openpyxl',
    'google-generativeai'
]

result = subprocess.run(
    ['pip', 'install', '-q'] + packages,
    capture_output=True, text=True
)

if result.returncode != 0:
    print(f"❌ Installation error: {result.stderr}")
else:
    print("✅ All dependencies installed!")

📦 Installing dependencies...
✅ All dependencies installed!


In [ ]:
# ============================================================================
# Cell 2: Setup & Imports (with status checks)
# ============================================================================
print("🔧 Setting up...\n")

# Standard libraries
try:
    import os, json, re, time, pandas as pd
    from typing import TypedDict, Annotated, Sequence, List, Dict, Any
    from pathlib import Path
    print("✅ Standard libraries loaded")
except Exception as e:
    print(f"❌ Standard libraries error: {e}")

# Google Colab
try:
    from google.colab import userdata, files
    print("✅ Google Colab loaded")
except Exception as e:
    print(f"❌ Google Colab error: {e}")

# LangChain
try:
    from langchain_google_genai import ChatGoogleGenerativeAI
    from langchain_core.messages import BaseMessage, HumanMessage
    print("✅ LangChain loaded")
except Exception as e:
    print(f"❌ LangChain error: {e}")

# LangGraph
try:
    from langgraph.graph import StateGraph, START, END
    from langgraph.graph.message import add_messages
    print("✅ LangGraph loaded")
except Exception as e:
    print(f"❌ LangGraph error: {e}")

# API Key
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    key = os.environ.get("GOOGLE_API_KEY", "")
    if key and len(key) > 10:
        print(f"✅ API Key loaded (starts with: {key[:8]}...)")
    else:
        print("❌ API Key is empty or too short!")
except Exception as e:
    print(f"❌ API Key error: {e}")

# Initialize LLM
llm = None
try:
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-preview-05-20", temperature=0.1)
    print("✅ Gemini 2.5 Flash initialized")
except Exception as e:
    print(f"❌ LLM initialization error: {e}")

print("\n" + "="*50)
print("Setup complete!" if llm else "Setup failed - check errors above")
print("="*50)

🔧 Setting up...

✅ Standard libraries loaded
✅ Google Colab loaded
✅ LangChain loaded
✅ LangGraph loaded
✅ API Key loaded (starts with: AIzaSyA0...)
✅ Gemini 2.5 Flash initialized

Setup complete!


In [ ]:
# ============================================================================
# Cell 3: Define State (LangGraph)
# ============================================================================
class MetadataState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    file_path: str
    file_type: str  # 'geo_txt', 'arrayexpress_csv', 'xlsx'
    raw_entries: List[Dict[str, Any]]
    extracted_metadata: List[Dict[str, Any]]
    output_path: str
    errors: List[Dict[str, Any]]
    verbose: bool

print("✅ State defined")

✅ State defined


In [ ]:
# ============================================================================
# Cell 4: File Parsers (GEO, ArrayExpress, Excel) - UPDATED
# ============================================================================
import csv

def parse_geo_txt(file_path: str, verbose: bool = False) -> List[Dict[str, Any]]:
    """
    Parse GEO search result .txt file (numbered format).
    Handles tab-separated fields like: Organism:\tMus musculus
    """
    entries = []

    if verbose:
        print(f"  📂 Opening: {file_path}")

    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        if verbose:
            print(f"  📏 File size: {len(content):,} characters")
    except Exception as e:
        print(f"  ❌ Error reading file: {e}")
        return entries

    # Split by numbered entries (e.g., "1. Title here")
    blocks = re.split(r'\n(?=\d+\.\s)', content)
    if verbose:
        print(f"  📦 Found {len(blocks)} text blocks")

    parsed = 0
    skipped = 0

    for block in blocks:
        block = block.strip()
        if not block:
            continue

        # Only process Series entries (have GSE accession)
        if 'Series' not in block or 'GSE' not in block:
            skipped += 1
            continue

        try:
            # Entry number and title
            title_match = re.match(r'(\d+)\.\s+(.+?)\n', block)
            if not title_match:
                skipped += 1
                continue

            entry_num = title_match.group(1)
            title = title_match.group(2).strip()

            # Description
            desc_match = re.search(r'\(Submitter supplied\)\s*(.+?)(?=\nOrganism:)', block, re.DOTALL)
            description = desc_match.group(1).strip()[:2000] if desc_match else ''

            # Organism (handles tabs)
            org_match = re.search(r'Organism:[\t\s]+(.+?)(?=\n)', block)
            organism = org_match.group(1).strip() if org_match else ''

            # Type (handles tabs)
            type_match = re.search(r'Type:[\t\s]+(.+?)(?=\n)', block)
            study_type = type_match.group(1).strip() if type_match else ''

            # Platform and sample count
            platform_match = re.search(r'Platform:\s*(GPL\d+)\s*(\d+)\s*Samples?', block)
            if platform_match:
                platform = platform_match.group(1)
                sample_count = platform_match.group(2)
            else:
                platform_match2 = re.search(r'Platform:\s*(GPL\d+)', block)
                platform = platform_match2.group(1) if platform_match2 else ''
                sample_match = re.search(r'(\d+)\s*Samples?', block)
                sample_count = sample_match.group(1) if sample_match else ''

            # Accession
            acc_match = re.search(r'Accession:\s*(GSE\d+)', block)
            accession = acc_match.group(1) if acc_match else ''

            if accession:
                entries.append({
                    'entry_num': entry_num,
                    'title': title,
                    'description': description,
                    'organism': organism,
                    'study_type': study_type,
                    'platform': platform,
                    'sample_count': sample_count,
                    'accession': accession,
                    'source': 'GEO'
                })
                parsed += 1
            else:
                skipped += 1

        except Exception as e:
            skipped += 1
            if verbose:
                print(f"  ⚠️ Parse error in block: {e}")

    if verbose:
        print(f"  ✅ Parsed: {parsed} entries | Skipped: {skipped}")

    return entries


def parse_geo_tsv(file_path: str, verbose: bool = False) -> List[Dict[str, Any]]:
    """
    Parse GEO curated datasets TSV file (tab-separated with headers).
    Expected columns: GEO_Accession, Title, Summary, FTPLink, Included

    NOTE: This format doesn't include organism/platform - use enrich_from_geo()
    to fetch missing metadata from GEO API.
    """
    entries = []

    if verbose:
        print(f"  📂 Opening TSV: {file_path}")

    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            # Use csv.DictReader for robust TSV parsing
            reader = csv.DictReader(f, delimiter='\t')

            # Validate expected columns
            expected_cols = {'GEO_Accession', 'Title', 'Summary'}
            if not expected_cols.issubset(set(reader.fieldnames or [])):
                if verbose:
                    print(f"  ⚠️ Missing expected columns. Found: {reader.fieldnames}")
                return []

            for idx, row in enumerate(reader):
                accession = row.get('GEO_Accession', '').strip()

                # Skip non-GSE entries or empty rows
                if not accession:
                    continue

                # Handle GDS references in older entries (e.g., "<GDS>345</GDS>")
                if accession.startswith('<GDS>'):
                    continue

                if not accession.startswith('GSE'):
                    continue

                # Clean summary (remove surrounding quotes and HTML entities)
                summary = row.get('Summary', '')
                if summary.startswith('"') and summary.endswith('"'):
                    summary = summary[1:-1]
                # Clean common HTML entities
                summary = summary.replace('&apos;', "'").replace('&quot;', '"')
                summary = summary.replace('&lt;', '<').replace('&gt;', '>')
                summary = summary.replace('&amp;', '&')

                entries.append({
                    'entry_num': str(idx + 1),
                    'accession': accession,
                    'title': row.get('Title', '').strip(),
                    'description': summary[:2000],
                    'ftp_link': row.get('FTPLink', ''),
                    'included': row.get('Included', ''),
                    # These fields are NOT in the TSV - will be empty or enriched later
                    'organism': '',
                    'study_type': '',
                    'platform': '',
                    'sample_count': '',
                    'source': 'GEO'
                })

        if verbose:
            print(f"  ✅ Parsed: {len(entries)} entries from TSV")

    except Exception as e:
        print(f"  ❌ Error parsing TSV: {e}")
        import traceback
        traceback.print_exc()

    return entries


def parse_arrayexpress_csv(file_path: str, verbose: bool = False) -> List[Dict[str, Any]]:
    """
    Parse ArrayExpress/BioStudies CSV file.
    """
    entries = []
    df = None

    if verbose:
        print(f"  📂 Opening: {file_path}")

    # Try different encodings
    encodings = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            if verbose:
                print(f"  📖 Loaded with encoding: {encoding}")
            break
        except:
            continue

    if df is None:
        print(f"  ❌ Could not read CSV with any encoding")
        return entries

    if verbose:
        print(f"  📏 Rows: {len(df)} | Columns: {list(df.columns)[:5]}...")

    for idx, row in df.iterrows():
        entry = {
            'entry_num': str(idx + 1),
            'accession': str(row.get('accession', row.get('study_id', ''))),
            'title': str(row.get('title', '')),
            'description': str(row.get('description', ''))[:2000],
            'organism': str(row.get('organism', '')),
            'study_type': str(row.get('study_type', row.get('technology', ''))),
            'platform': str(row.get('technology', '')),
            'sample_count': str(row.get('sample_count', '')),
            'experimental_factors': str(row.get('experimental_factors', '')),
            'experimental_design': str(row.get('experimental_design', '')),
            'source': 'ArrayExpress'
        }
        # Clean up 'nan' strings
        entry = {k: ('' if v == 'nan' else v) for k, v in entry.items()}
        entries.append(entry)

    if verbose:
        print(f"  ✅ Parsed: {len(entries)} entries")

    return entries


def parse_xlsx(file_path: str, verbose: bool = False) -> List[Dict[str, Any]]:
    """
    Parse Excel file (.xlsx, .xls).
    """
    entries = []

    if verbose:
        print(f"  📂 Opening: {file_path}")

    try:
        df = pd.read_excel(file_path)
        if verbose:
            print(f"  📏 Rows: {len(df)} | Columns: {list(df.columns)[:5]}...")
    except Exception as e:
        print(f"  ❌ Error reading Excel: {e}")
        return entries

    for idx, row in df.iterrows():
        entry = {col: str(row[col]) if pd.notna(row[col]) else '' for col in df.columns}
        entry['entry_num'] = str(idx + 1)
        entry['source'] = 'Excel'
        entries.append(entry)

    if verbose:
        print(f"  ✅ Parsed: {len(entries)} entries")

    return entries


def detect_and_parse(file_path: str, verbose: bool = False) -> tuple:
    """
    Auto-detect file type and parse accordingly.
    Returns: (file_type, entries)

    Supported formats:
    - .txt (GEO numbered search results OR tab-separated curated datasets)
    - .csv (ArrayExpress)
    - .xlsx/.xls (Excel)
    """
    path = Path(file_path)
    suffix = path.suffix.lower()

    if verbose:
        print(f"\n🔍 Detected file extension: {suffix}")

    if suffix == '.txt':
        # Peek at file to determine format
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            first_line = f.readline()
            second_line = f.readline()

        # Check if it's a TSV (curated datasets format)
        # Look for tab-separated header with GEO_Accession
        if '\t' in first_line and 'GEO_Accession' in first_line:
            if verbose:
                print("  📋 Format: GEO curated datasets (TSV)")
            return 'geo_tsv', parse_geo_tsv(file_path, verbose)

        # Check if it's numbered GEO search results
        elif re.match(r'\d+\.\s+', first_line) or re.match(r'\d+\.\s+', second_line):
            if verbose:
                print("  📋 Format: GEO numbered search results")
            return 'geo_txt', parse_geo_txt(file_path, verbose)

        # Default to trying GEO txt parser
        else:
            if verbose:
                print("  📋 Format: Unknown .txt, trying GEO parser...")
            entries = parse_geo_txt(file_path, verbose)
            if entries:
                return 'geo_txt', entries
            # If that fails, try TSV
            entries = parse_geo_tsv(file_path, verbose)
            if entries:
                return 'geo_tsv', entries
            return 'unknown', []

    elif suffix == '.csv':
        return 'arrayexpress_csv', parse_arrayexpress_csv(file_path, verbose)

    elif suffix in ['.xlsx', '.xls']:
        return 'xlsx', parse_xlsx(file_path, verbose)

    else:
        print(f"  ❌ Unsupported file type: {suffix}")
        print(f"  Supported: .txt (GEO), .csv (ArrayExpress), .xlsx (Excel)")
        raise ValueError(f"Unsupported file type: {suffix}")

print("✅ Parsers loaded (GEO TXT, GEO TSV, ArrayExpress, Excel)")

✅ Parsers loaded (GEO TXT, GEO TSV, ArrayExpress, Excel)


In [ ]:
# ============================================================================
# Cell 4b: GEO API Enrichment (OPTIONAL - for TSV files missing metadata)
# ============================================================================
import urllib.request
import urllib.error
import xml.etree.ElementTree as ET

def fetch_geo_metadata(accession: str, verbose: bool = False) -> Dict[str, str]:
    """
    Fetch metadata from GEO for a single accession using Entrez API.
    Returns dict with organism, platform, sample_count, study_type.
    """
    result = {
        'organism': '',
        'platform': '',
        'sample_count': '',
        'study_type': ''
    }

    if not accession or not accession.startswith('GSE'):
        return result

    # GEO eSummary API
    url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=gds&term={accession}[Accession]&retmax=1"

    try:
        # First get the GEO ID
        with urllib.request.urlopen(url, timeout=10) as response:
            search_xml = response.read().decode('utf-8')

        # Parse to get ID
        root = ET.fromstring(search_xml)
        id_elem = root.find('.//Id')
        if id_elem is None:
            return result

        geo_id = id_elem.text

        # Now fetch summary
        summary_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=gds&id={geo_id}"
        with urllib.request.urlopen(summary_url, timeout=10) as response:
            summary_xml = response.read().decode('utf-8')

        root = ET.fromstring(summary_xml)

        # Extract fields from summary
        for item in root.findall('.//Item'):
            name = item.get('Name', '')
            text = item.text or ''

            if name == 'taxon':
                result['organism'] = text
            elif name == 'GPL':
                result['platform'] = f"GPL{text}" if text and not text.startswith('GPL') else text
            elif name == 'n_samples':
                result['sample_count'] = text
            elif name == 'gdsType':
                result['study_type'] = text

        if verbose:
            print(f"    📡 Fetched: {accession} -> {result['organism']}")

    except Exception as e:
        if verbose:
            print(f"    ⚠️ API error for {accession}: {e}")

    return result


def enrich_entries_from_geo(entries: List[Dict], verbose: bool = False,
                            delay: float = 0.35, max_entries: int = None) -> List[Dict]:
    """
    Enrich entries with metadata from GEO API.

    Args:
        entries: List of parsed entries
        verbose: Show progress
        delay: Seconds between API calls (NCBI recommends max 3/sec)
        max_entries: Limit enrichment to first N entries

    Returns:
        Enriched entries list
    """
    print(f"\n🔬 Enriching {len(entries)} entries from GEO API...")
    print(f"   (This may take a while - ~{len(entries) * delay / 60:.1f} minutes)")

    enriched = []
    to_process = entries[:max_entries] if max_entries else entries

    for i, entry in enumerate(to_process):
        acc = entry.get('accession', '')

        # Only fetch if missing key metadata
        if not entry.get('organism') or not entry.get('platform'):
            geo_data = fetch_geo_metadata(acc, verbose=verbose)

            # Update entry with fetched data (only if field is empty)
            for key in ['organism', 'platform', 'sample_count', 'study_type']:
                if not entry.get(key) and geo_data.get(key):
                    entry[key] = geo_data[key]

            time.sleep(delay)  # Rate limiting

        enriched.append(entry)

        # Progress
        if (i + 1) % 50 == 0:
            print(f"   Progress: {i+1}/{len(to_process)}")

    print(f"   ✅ Enrichment complete!")
    return enriched

print("✅ GEO API enrichment functions ready")
print("\n📖 Usage (optional):")
print("   entries = enrich_entries_from_geo(entries, verbose=True)")

✅ GEO API enrichment functions ready

📖 Usage (optional):
   entries = enrich_entries_from_geo(entries, verbose=True)


In [ ]:
# ============================================================================
# Cell 5: LLM Extraction Function (with error handling) - UPDATED
# ============================================================================

def extract_metadata_llm(entry: Dict[str, Any], verbose: bool = False) -> Dict[str, Any]:
    """
    Use Gemini to extract structured metadata from study entry.
    Returns extracted fields or defaults on error.
    """

    default_result = {
        "sequencing_approach": "",
        "is_single_cell": "",
        "tissue_cell_type": "",
        "compound_treatment": "",
        "dose": "",
        "time_points": "",
        "disease_condition": ""
    }

    # Build context from entry fields
    context_parts = []
    fields_to_use = ['title', 'description', 'organism', 'study_type', 'platform',
                     'experimental_factors', 'experimental_design']

    for key in fields_to_use:
        val = entry.get(key, '')
        if val and str(val) not in ['', 'nan', 'None', 'NaN']:
            context_parts.append(f"{key}: {val}")

    context = "\n".join(context_parts)

    if verbose:
        print(f"    📝 Context: {len(context)} chars")

    # UPDATED: Lower threshold since TSV entries may have less metadata
    if len(context) < 10:
        if verbose:
            print(f"    ⚠️ Context too short, skipping LLM")
        return default_result

    prompt = f"""You are an expert bioinformatician. Extract metadata from this study entry.

STUDY ENTRY:
{context}

Extract these fields (use empty string "" if not found, do NOT guess):

1. sequencing_approach: One of ["microarray", "bulk RNA-seq", "scRNA-seq", "snRNA-seq", "ATAC-seq", "ChIP-seq", "Hi-C", "4C-seq", "multiome", "whole genome sequencing", "proteomics", "metabolomics", "other"]
2. is_single_cell: "yes" if single-cell or single-nucleus, "no" if bulk, "" if unclear
3. tissue_cell_type: The tissue, organ, or cell type (e.g., "liver", "CD8+ T cells", "organoids")
4. compound_treatment: Chemical compound, drug, or treatment (e.g., "dexamethasone", "BPA", "high glucose")
5. dose: Dosage/concentration if mentioned (e.g., "100 nM", "10 mg/kg")
6. time_points: Time points if mentioned (e.g., "24h", "7 days", "1h")
7. disease_condition: Disease or condition studied (e.g., "obesity", "cancer", "diabetes")

CRITICAL: Return ONLY valid JSON. No markdown, no explanation, no code blocks.

{{"sequencing_approach": "", "is_single_cell": "", "tissue_cell_type": "", "compound_treatment": "", "dose": "", "time_points": "", "disease_condition": ""}}"""

    # Check if LLM is available
    if llm is None:
        print(f"    ❌ LLM not initialized!")
        return default_result

    try:
        if verbose:
            print(f"    🤖 Calling Gemini...", end=" ")

        response = llm.invoke([HumanMessage(content=prompt)])

        if verbose:
            print("received response")

        text = response.content.strip()

        if verbose:
            print(f"    📄 Raw response ({len(text)} chars): {text[:100]}...")

        # Clean markdown code blocks if present
        if '```' in text:
            # Extract JSON from code block
            match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
            if match:
                text = match.group(0)
            else:
                # Try removing code block markers
                text = re.sub(r'```json\s*', '', text)
                text = re.sub(r'```\s*', '', text)
                text = text.strip()

        result = json.loads(text)

        if verbose:
            print(f"    ✅ Parsed JSON: {list(result.keys())}")

        return result

    except json.JSONDecodeError as e:
        print(f"    ❌ JSON Parse Error: {e}")
        if verbose:
            print(f"    Raw text: {text[:200]}...")
        return default_result

    except Exception as e:
        error_type = type(e).__name__
        print(f"    ❌ {error_type}: {e}")
        return default_result

print("✅ LLM extraction function ready")

✅ LLM extraction function ready


In [ ]:
# ============================================================================
# Cell 6: LangGraph Nodes
# ============================================================================

def load_node(state: MetadataState) -> MetadataState:
    """Node 1: Load and parse input file."""
    file_path = state['file_path']
    verbose = state.get('verbose', False)

    print(f"\n📂 [LOAD NODE]")
    print(f"   File: {file_path}")

    try:
        file_type, entries = detect_and_parse(file_path, verbose=verbose)
        print(f"   ✅ Loaded {len(entries)} entries (type: {file_type})")
        return {**state, 'file_type': file_type, 'raw_entries': entries}
    except Exception as e:
        print(f"   ❌ Load failed: {e}")
        return {**state, 'file_type': 'error', 'raw_entries': [], 'errors': [{'stage': 'load', 'error': str(e)}]}


def extract_node(state: MetadataState) -> MetadataState:
    """Node 2: Extract metadata using LLM."""
    entries = state.get('raw_entries', [])
    verbose = state.get('verbose', False)
    errors = state.get('errors', [])

    print(f"\n🧠 [EXTRACT NODE]")

    if not entries:
        print(f"   ⚠️ No entries to process")
        return {**state, 'extracted_metadata': []}

    print(f"   Processing {len(entries)} entries...\n")

    all_metadata = []

    for i, entry in enumerate(entries):
        acc = entry.get('accession', f"entry_{i+1}")
        print(f"   [{i+1}/{len(entries)}] {acc}...", end=" ")

        try:
            # Call LLM
            llm_data = extract_metadata_llm(entry, verbose=verbose)

            # Combine parsed + LLM extracted
            metadata = {
                'accession': entry.get('accession', ''),
                'title': entry.get('title', ''),
                'organism': entry.get('organism', ''),
                'platform': entry.get('platform', ''),
                'sample_count': entry.get('sample_count', ''),
                'study_type': entry.get('study_type', ''),
                'source_db': entry.get('source', ''),
                'sequencing_approach': llm_data.get('sequencing_approach', ''),
                'is_single_cell': llm_data.get('is_single_cell', ''),
                'tissue_cell_type': llm_data.get('tissue_cell_type', ''),
                'compound_treatment': llm_data.get('compound_treatment', ''),
                'dose': llm_data.get('dose', ''),
                'time_points': llm_data.get('time_points', ''),
                'disease_condition': llm_data.get('disease_condition', '')
            }

            all_metadata.append(metadata)
            print("✅")

        except Exception as e:
            print(f"❌ {e}")
            errors.append({'accession': acc, 'error': str(e)})

        # Rate limiting
        time.sleep(0.4)

        # Progress checkpoint every 25
        if (i + 1) % 25 == 0:
            print(f"\n   💾 Progress: {i+1}/{len(entries)} done\n")

    return {**state, 'extracted_metadata': all_metadata, 'errors': errors}


def save_node(state: MetadataState) -> MetadataState:
    """Node 3: Save results to CSV."""
    metadata = state.get('extracted_metadata', [])
    output_path = state.get('output_path', 'metadata_output.csv')
    errors = state.get('errors', [])

    print(f"\n💾 [SAVE NODE]")

    if not metadata:
        print(f"   ⚠️ No metadata to save")
        return state

    try:
        df = pd.DataFrame(metadata)

        # Reorder columns
        column_order = [
            'accession', 'title', 'organism', 'platform', 'sample_count',
            'study_type', 'sequencing_approach', 'is_single_cell',
            'tissue_cell_type', 'compound_treatment', 'dose', 'time_points',
            'disease_condition', 'source_db'
        ]
        df = df[[col for col in column_order if col in df.columns]]

        df.to_csv(output_path, index=False)
        print(f"   ✅ Saved {len(df)} entries to: {output_path}")

        if errors:
            print(f"   ⚠️ {len(errors)} errors occurred")

    except Exception as e:
        print(f"   ❌ Save failed: {e}")

    return state

print("✅ LangGraph nodes defined")

✅ LangGraph nodes defined


In [ ]:
# ============================================================================
# Cell 7: Create LangGraph Agent
# ============================================================================

def create_metadata_agent():
    """Create the LangGraph agent with load -> extract -> save flow."""
    graph = StateGraph(MetadataState)

    # Add nodes
    graph.add_node("load", load_node)
    graph.add_node("extract", extract_node)
    graph.add_node("save", save_node)

    # Define edges
    graph.add_edge(START, "load")
    graph.add_edge("load", "extract")
    graph.add_edge("extract", "save")
    graph.add_edge("save", END)

    return graph.compile()

agent = create_metadata_agent()
print("✅ LangGraph Agent ready!")
print("   Flow: load → extract → save")

✅ LangGraph Agent ready!
   Flow: load → extract → save


In [ ]:
# ============================================================================
# Cell 8: Main Extraction Function
# ============================================================================

def extract_metadata(file_path: str, output_path: str = None, limit: int = None, verbose: bool = False):
    """
    Extract metadata from GEO/ArrayExpress/Excel file.

    Args:
        file_path: Path to input file (.txt, .csv, .xlsx)
        output_path: Output CSV path (auto-generated if None)
        limit: Max entries to process (for testing)
        verbose: Show detailed progress

    Returns:
        DataFrame with extracted metadata
    """

    # Auto-generate output path
    if output_path is None:
        input_name = Path(file_path).stem
        output_path = f"{input_name}_metadata_extracted.csv"

    print("\n" + "="*60)
    print("  🧬 METADATA EXTRACTION AGENT")
    print("="*60)
    print(f"  Input:  {file_path}")
    print(f"  Output: {output_path}")
    if limit:
        print(f"  Limit:  {limit} entries")
    print(f"  Verbose: {verbose}")
    print("="*60)

    # If limit specified, parse first then limit
    if limit:
        print(f"\n⚠️ Limiting to first {limit} entries")
        file_type, all_entries = detect_and_parse(file_path, verbose=verbose)
        entries = all_entries[:limit]
        print(f"   Total in file: {len(all_entries)} | Processing: {len(entries)}")

        # Run with limited entries
        initial_state = {
            'messages': [HumanMessage(content=f"Extract metadata from {file_path}")],
            'file_path': file_path,
            'file_type': file_type,
            'raw_entries': entries,
            'extracted_metadata': [],
            'output_path': output_path,
            'errors': [],
            'verbose': verbose
        }

        # Skip load node, go to extract
        result = extract_node(initial_state)
        result = save_node(result)
    else:
        # Run full agent
        initial_state = {
            'messages': [HumanMessage(content=f"Extract metadata from {file_path}")],
            'file_path': file_path,
            'file_type': '',
            'raw_entries': [],
            'extracted_metadata': [],
            'output_path': output_path,
            'errors': [],
            'verbose': verbose
        }
        result = agent.invoke(initial_state)

    # Summary
    print("\n" + "="*60)
    print("  ✅ EXTRACTION COMPLETE")
    print("="*60)

    extracted = result.get('extracted_metadata', [])
    errors = result.get('errors', [])

    print(f"  Extracted: {len(extracted)} entries")
    print(f"  Errors:    {len(errors)}")
    print(f"  Output:    {output_path}")

    if errors and verbose:
        print("\n  Error details:")
        for e in errors[:5]:
            print(f"    - {e}")

    print("="*60 + "\n")

    if extracted:
        return pd.DataFrame(extracted)
    return pd.DataFrame()

print("✅ Main function ready!")
print("\n📖 Usage:")
print("   df = extract_metadata('file.txt', limit=5, verbose=True)")

✅ Main function ready!

📖 Usage:
   df = extract_metadata('file.txt', limit=5, verbose=True)


In [ ]:
# ============================================================================
# Cell 9: Batch Processing for Large Files
# ============================================================================

def batch_extract_metadata(
    file_path: str,
    output_path: str = None,
    batch_size: int = 100,
    start_index: int = 0,
    max_entries: int = None,
    verbose: bool = False
):
    """
    Process large files in batches with checkpointing.
    Use this for files with 1000+ entries.

    Args:
        file_path: Input file path
        output_path: Output CSV path
        batch_size: Entries per batch (default 100)
        start_index: Resume from this index
        max_entries: Maximum entries to process
        verbose: Detailed output
    """

    if output_path is None:
        input_name = Path(file_path).stem
        output_path = f"{input_name}_metadata_extracted.csv"

    print("\n" + "="*60)
    print("  🧬 BATCH METADATA EXTRACTION")
    print("="*60)

    # Parse all entries
    print(f"\n📂 Loading file...")
    file_type, all_entries = detect_and_parse(file_path, verbose=verbose)
    print(f"   Total entries: {len(all_entries)}")

    # Apply limits
    if max_entries:
        entries = all_entries[start_index:start_index + max_entries]
    else:
        entries = all_entries[start_index:]

    print(f"   Processing: {len(entries)} entries (from index {start_index})")
    print(f"   Batch size: {batch_size}")

    all_metadata = []
    all_errors = []

    # Process in batches
    num_batches = (len(entries) + batch_size - 1) // batch_size

    for batch_num in range(num_batches):
        batch_start = batch_num * batch_size
        batch_end = min(batch_start + batch_size, len(entries))
        batch = entries[batch_start:batch_end]

        global_start = start_index + batch_start + 1
        global_end = start_index + batch_end

        print(f"\n📦 Batch {batch_num + 1}/{num_batches} (entries {global_start}-{global_end})")

        for i, entry in enumerate(batch):
            acc = entry.get('accession', f"entry_{batch_start + i + 1}")
            print(f"   [{global_start + i}] {acc}...", end=" ")

            try:
                llm_data = extract_metadata_llm(entry, verbose=verbose)

                metadata = {
                    'accession': entry.get('accession', ''),
                    'title': entry.get('title', ''),
                    'organism': entry.get('organism', ''),
                    'platform': entry.get('platform', ''),
                    'sample_count': entry.get('sample_count', ''),
                    'study_type': entry.get('study_type', ''),
                    'source_db': entry.get('source', ''),
                    **llm_data
                }

                all_metadata.append(metadata)
                print("✅")

            except Exception as e:
                print(f"❌ {e}")
                all_errors.append({'accession': acc, 'error': str(e)})

            time.sleep(0.4)

        # Save checkpoint after each batch
        if all_metadata:
            df = pd.DataFrame(all_metadata)
            df.to_csv(output_path, index=False)
            print(f"   💾 Checkpoint: {len(all_metadata)} entries saved to {output_path}")

    # Final summary
    print("\n" + "="*60)
    print("  ✅ BATCH EXTRACTION COMPLETE")
    print("="*60)
    print(f"  Extracted: {len(all_metadata)} entries")
    print(f"  Errors:    {len(all_errors)}")
    print(f"  Output:    {output_path}")
    print("="*60 + "\n")

    return pd.DataFrame(all_metadata) if all_metadata else pd.DataFrame()

print("✅ Batch processor ready!")
print("\n📖 Usage:")
print("   df = batch_extract_metadata('file.txt', batch_size=100)")

✅ Batch processor ready!

📖 Usage:
   df = batch_extract_metadata('file.txt', batch_size=100)


In [ ]:
# ============================================================================
# Cell 10: Upload File
# ============================================================================
print("📤 Upload your file:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {filename}")

In [ ]:
filename='/content/GEO_9_part1.txt'

In [ ]:
# ============================================================================
# Cell 11: Test Parser
# ============================================================================
print("🔍 Testing parser...\n")

file_type, entries = detect_and_parse(filename, verbose=True)

print(f"\n" + "="*50)
print(f"Found {len(entries)} entries")
print("="*50)

if entries:
    print("\nFirst 2 entries:")
    for i, e in enumerate(entries[:2]):
        print(f"\n[{i+1}] {e.get('accession', 'N/A')}")
        print(f"    Title: {e.get('title', '')[:60]}...")
        print(f"    Organism: {e.get('organism', 'N/A')}")
        print(f"    Type: {e.get('study_type', 'N/A')}")
else:
    print("\n❌ No entries found! Check file format.")

🔍 Testing parser...


🔍 Detected file extension: .txt
  📋 Format: GEO numbered search results
  📂 Opening: /content/GEO_9_part1.txt
  📏 File size: 2,163,355 characters
  📦 Found 2557 text blocks
  ✅ Parsed: 2542 entries | Skipped: 14

Found 2542 entries

First 2 entries:

[1] GSE315771
    Title: HDI-STARR-seq library profiling of differential accessible c...
    Organism: Mus musculus
    Type: Other

[2] GSE315532
    Title: Transcriptomic Profiling Reveals Thyroid Hormone-Mediated an...
    Organism: Rattus norvegicus
    Type: Expression profiling by high throughput sequencing


In [ ]:
# Test the parser
print("🔍 Testing parser...\n")

file_type, entries = detect_and_parse(filename, verbose=True)

print(f"\n" + "="*50)
print(f"Found {len(entries)} entries (type: {file_type})")
print("="*50)

if entries:
    print("\nFirst 2 entries:")
    for i, e in enumerate(entries[:2]):
        print(f"\n[{i+1}] {e.get('accession', 'N/A')}")
        print(f"    Title: {e.get('title', '')[:60]}...")
        print(f"    Description: {e.get('description', '')[:80]}...")
        print(f"    Organism: {e.get('organism', 'N/A') or '(not in TSV - use enrichment)'}")
else:
    print("\n❌ No entries found! Check file format.")

🔍 Testing parser...


🔍 Detected file extension: .txt
  📋 Format: GEO curated datasets (TSV)
  📂 Opening TSV: /content/curatedDatasets_part2.txt
  ✅ Parsed: 1079 entries from TSV

Found 1079 entries (type: geo_tsv)

First 2 entries:

[1] GSE155770
    Title: Sevoflurane Exposure Alters Long Non-coding RNA and Messenge...
    Description: The focus of current study is to investigate the effect of developing sevofluran...
    Organism: (not in TSV - use enrichment)

[2] GSE164552
    Title: IL-22 promotes formation of MUC17 glycocalyx barrier in post...
    Description: The intestine is under constant exposure to chemicals, antigens and microorganis...
    Organism: (not in TSV - use enrichment)


In [ ]:
# Fix: Update to correct model name
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)
print("✅ LLM updated to gemini-2.5-flash")

✅ LLM updated to gemini-2.5-flash


In [ ]:
# ============================================================================
# Cell 12: Test LLM on ONE entry
# ============================================================================
if entries:
    print("🧪 Testing LLM on first entry...\n")
    print(f"Entry: {entries[0].get('accession')}")
    print(f"Title: {entries[0].get('title', '')[:70]}...\n")

    result = extract_metadata_llm(entries[0], verbose=True)

    print("\n" + "="*40)
    print("Extracted:")
    for k, v in result.items():
        print(f"  {k}: {v}")
else:
    print("❌ No entries - run Cell 11 first")

🧪 Testing LLM on first entry...

Entry: GSE155770
Title: Sevoflurane Exposure Alters Long Non-coding RNA and Messenger RNA Prof...

    📝 Context: 563 chars
    🤖 Calling Gemini... received response
    📄 Raw response (202 chars): {"sequencing_approach": "microarray", "is_single_cell": "no", "tissue_cell_type": "hippocampus", "co...
    ✅ Parsed JSON: ['sequencing_approach', 'is_single_cell', 'tissue_cell_type', 'compound_treatment', 'dose', 'time_points', 'disease_condition']

Extracted:
  sequencing_approach: microarray
  is_single_cell: no
  tissue_cell_type: hippocampus
  compound_treatment: sevoflurane
  dose: 
  time_points: 
  disease_condition: neurotoxicity


In [ ]:
# ============================================================================
# Cell 13: Extract (5 entries test)
# ============================================================================
df = extract_metadata(filename, limit=5, verbose=True)
df

In [ ]:
# ============================================================================
# Cell 14: Full extraction (uncomment when ready)
# ============================================================================

  #For small files (<100 entries):
df = extract_metadata(filename, verbose=False)

#For large files (100+ entries):
df = batch_extract_metadata(filename, batch_size=1000, verbose=False)


  🧬 METADATA EXTRACTION AGENT
  Input:  /content/GEO_9_part2.txt
  Output: GEO_9_part2_metadata_extracted.csv
  Verbose: False

📂 [LOAD NODE]
   File: /content/GEO_9_part2.txt
   ✅ Loaded 438 entries (type: geo_txt)

🧠 [EXTRACT NODE]
   Processing 438 entries...

   [1/438] GSE22868... ✅
   [2/438] GSE31738... ✅
   [3/438] GSE28999... ✅
   [4/438] GSE28677... ✅
   [5/438] GSE29513... ✅
   [6/438] GSE31503... ✅
   [7/438] GSE31411... ✅
   [8/438] GSE31131... ✅
   [9/438] GSE29976... ✅
   [10/438] GSE30914... ✅
   [11/438] GSE30646... ✅
   [12/438] GSE30638... ✅
   [13/438] GSE22941... ✅
   [14/438] GSE26515... ✅
   [15/438] GSE27132... ✅
   [16/438] GSE30261... ✅
   [17/438] GSE23876... ✅
   [18/438] GSE25245... ✅
   [19/438] GSE22218... ✅
   [20/438] GSE28488... ✅
   [21/438] GSE29264... ✅
   [22/438] GSE29132... ✅
   [23/438] GSE29015... ✅
   [24/438] GSE27263... ✅
   [25/438] GSE28896... ✅

   💾 Progress: 25/438 done

   [26/438] GSE25196... ✅
   [27/438] GSE28532... ✅
   [28/438] G

In [ ]:
# ============================================================================
# Cell 15: Download results
# ============================================================================

# files.download('your_output_file.csv')

In [ ]:
# ============================================================================
# File Splitter for TSV/TXT files
# ============================================================================
import csv
import math

def split_file(filename, num_parts=5, output_prefix=None):
    """
    Split a TSV/TXT file into multiple parts.

    Args:
        filename: Input file path
        num_parts: Number of parts to split into
        output_prefix: Prefix for output files (default: original filename)

    Returns:
        List of created filenames
    """
    if output_prefix is None:
        output_prefix = filename.replace('.txt', '').replace('.csv', '')

    # Read all lines
    with open(filename, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    # Check if first line is header
    header = lines[0] if '\t' in lines[0] and 'GEO_Accession' in lines[0] else None
    data_lines = lines[1:] if header else lines

    total_entries = len(data_lines)
    entries_per_part = math.ceil(total_entries / num_parts)

    print(f"📂 Input: {filename}")
    print(f"📊 Total entries: {total_entries}")
    print(f"📦 Splitting into {num_parts} parts (~{entries_per_part} entries each)\n")

    created_files = []

    for i in range(num_parts):
        start_idx = i * entries_per_part
        end_idx = min(start_idx + entries_per_part, total_entries)

        # Skip if no data for this part
        if start_idx >= total_entries:
            break

        chunk = data_lines[start_idx:end_idx]
        part_filename = f"{output_prefix}_part{i+1}.txt"

        with open(part_filename, 'w', encoding='utf-8') as f:
            # Write header if exists
            if header:
                f.write(header)
            f.writelines(chunk)

        created_files.append(part_filename)
        print(f"   ✅ {part_filename}: {len(chunk)} entries (rows {start_idx+1}-{end_idx})")

    print(f"\n🎉 Done! Created {len(created_files)} files")
    return created_files

# ============================================================================
# Usage
# ============================================================================

# Split into 5 parts
files = split_file(filename, num_parts=5)


📂 Input: /content/GEO_9.txt
📊 Total entries: 106465
📦 Splitting into 5 parts (~21293 entries each)

   ✅ /content/GEO_9_part1.txt: 21293 entries (rows 1-21293)
   ✅ /content/GEO_9_part2.txt: 21293 entries (rows 21294-42586)
   ✅ /content/GEO_9_part3.txt: 21293 entries (rows 42587-63879)
   ✅ /content/GEO_9_part4.txt: 21293 entries (rows 63880-85172)
   ✅ /content/GEO_9_part5.txt: 21293 entries (rows 85173-106465)

🎉 Done! Created 5 files
